<a href="https://colab.research.google.com/github/BASE-Laboratory/OpenImpala/blob/master/tutorials/02_digital_twin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 2: The Digital Twin -- Bridging 3D Imaging to 1D Models

**The Pain Point:** Researchers often have gigabytes of high-resolution 3D CT scans of battery electrodes. Meanwhile, continuum modeling teams (using tools like PyBaMM) just need a single JSON file of transport parameters to run device-level simulations. Bridging this gap usually requires a messy, manual pipeline of different software tools.

**The Solution:** OpenImpala automatically calculates the 3D anisotropy of a real scan and exports it directly to the industry-standard **BPX (Battery Parameter eXchange)** format, ready to be dropped into continuum models.

**In this tutorial we will:**
1. Download and load a real 3D TIFF image stack of a two-phase microstructure.
2. Compute the tortuosity in all three cardinal directions (X, Y, Z) to measure anisotropy (the effects of electrode calendering/compression).
3. **Visualize the 3D diffusion field** using the C++ core API and `yt`.
4. Export the data to BPX JSON.
5. **The Mic Drop:** Directly load that JSON into PyBaMM and simulate a battery discharge curve.

In [ ]:
# Install OpenImpala, PyBaMM, and visualization utilities
!pip install -q openimpala pybamm bpx tifffile matplotlib yt

In [ ]:
import os
import time
import json
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import tifffile

import openimpala as oi
from openimpala import core as oic
import pybamm

print(f"OpenImpala version {oi.__version__} loaded.")

# --- THE JUPYTER HPC FIX ---
# Keep AMReX and MPI alive for the entire notebook to prevent kernel crashes
global_session = oi.Session()
global_session.__enter__()

# --- THE COLAB DATA FIX ---
# Download the sample TIFF directly from the OpenImpala GitHub repo.
# This guarantees the notebook works on ephemeral Colab environments.
data_url = "https://raw.githubusercontent.com/BASE-Laboratory/OpenImpala/master/data/SampleData_2Phase_stack_3d_1bit.tif"
data_path = "SampleData_2Phase_stack_3d_1bit.tif"

if not os.path.exists(data_path):
    print("Downloading sample XRT CT scan...")
    urllib.request.urlretrieve(data_url, data_path)
    print("Download complete!")

## 1. Loading Real Tomography Data

OpenImpala's C++ backend has built-in readers for TIFF, RAW, and HDF5. However, when using the Python API, the most "Pythonic" workflow is to load your data using standard scientific libraries like `tifffile` or `h5py`, preprocess it (e.g., cropping, filtering), and then hand the NumPy array to OpenImpala.

In [ ]:
# Read the TIFF stack into a 3D NumPy array (Z, Y, X)
microstructure = tifffile.imread(data_path).astype(np.int32)

print(f"Loaded image shape (Z, Y, X): {microstructure.shape} ({microstructure.size / 1e6:.1f}M voxels)")
print(f"Unique phases present:        {np.unique(microstructure)}")

# Visualize the middle slice
z_mid = microstructure.shape[0] // 2
plt.figure(figsize=(5, 5), dpi=100)
plt.imshow(microstructure[z_mid, :, :], cmap='gray')
plt.title(f"Microstructure Slice at Z = {z_mid}", fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

## 2. Calculating Directional Tortuosity (Anisotropy)

Let's assume **Phase 1 is our pore space** (electrolyte). We want to know how easily lithium ions can diffuse through this space.

Manufactured electrodes are calendered (compressed), which crushes the pores in the through-thickness (Z) direction. We will compute the tortuosity in all three cardinal directions to measure this anisotropy.

In [ ]:
directions = ['x', 'y', 'z']
tau_results = {}

print("Analyzing Anisotropy...")
t_start_all = time.time()

# 1. Compute Volume Fraction
vf = oi.volume_fraction(microstructure, phase=1)
print(f"Pore Volume Fraction (Porosity): {vf.fraction:.4f}\n")

# 2. Compute Tortuosity for each direction
for d in directions:
    t0 = time.time()
    # Check percolation first to avoid solver errors on blocked domains
    perc = oi.percolation_check(microstructure, phase=1, direction=d)

    if perc.percolates:
        # max_grid_size=128 optimizes AMReX domain decomposition for laptop/Colab RAM
        res = oi.tortuosity(microstructure, phase=1, direction=d, solver="flexgmres", max_grid_size=128)
        tau_results[d.upper()] = res.tortuosity
        t_solve = time.time() - t0
        print(f"  {d.upper()}-Direction: Tau = {res.tortuosity:.4f} (Solved in {t_solve:.2f}s)")
    else:
        tau_results[d.upper()] = None
        print(f"  {d.upper()}-Direction: Does NOT percolate.")

t_total = time.time() - t_start_all
print(f"\nFull anisotropic characterization of {microstructure.size / 1e6:.1f}M voxels complete in {t_total:.2f} seconds.")

Let's plot the results. Notice if the material has a preferred transport direction!

In [ ]:
dirs = list(tau_results.keys())
taus = [v for v in tau_results.values() if v is not None]

fig, ax = plt.subplots(figsize=(6, 4), dpi=120)
bars = ax.bar(dirs, taus, color=['#1f77b4', '#ff7f0e', '#2ca02c'], width=0.5)
ax.axhline(y=1.0, color='r', linestyle='--', label='Ideal limit (Tau=1.0)')

ax.set_ylabel(r"Tortuosity Factor ($\tau$)", fontweight='bold')
ax.set_title("Directional Tortuosity (Anisotropy)", fontweight='bold')
ax.set_ylim(1.0, max(taus) * 1.15)
ax.legend()

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 0.02,
            f'{yval:.3f}', ha='center', va='bottom', fontweight='bold')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.show()

## 3. Visualizing the 3D Diffusion Field

By default, the high-level `oi.tortuosity()` function turns off 3D field output to save disk space and time. However, because OpenImpala is built on AMReX, it natively outputs standard AMReX plotfiles.

To see the actual diffusion pathways, we will drop down one layer to the `openimpala.core` API. This allows us to turn on the `write_plotfile=True` flag. Then, we will use **yt** (the gold standard for AMReX visualization) to plot a slice of the linear potential gradient!

In [ ]:
import glob
import yt

# Create a directory for our AMReX plotfiles
out_dir = "./plotfiles"
os.makedirs(out_dir, exist_ok=True)

print("Running core solver to generate 3D plotfile...")

# Convert our NumPy array into an OpenImpala VoxelImage
img = oi.VoxelImage.from_numpy(microstructure.copy(), max_grid_size=128)

# Instantiate the C++ HYPRE solver directly, with write_plotfile=True
solver = oic.TortuosityHypre(
    img,
    vf.fraction,               # Volume fraction (calculated earlier)
    1,                         # Phase ID
    oic.Direction.Z,           # Solve in Z direction
    oic.SolverType.FlexGMRES,  # HYPRE solver
    out_dir,                   # Output directory
    0.0, 1.0,                  # Boundary conditions (V_lo, V_hi)
    1,                         # Verbose level
    True                       # <-- TURN ON PLOTFILE!
)

# Execute the solve
solver.value()
print("Plotfile generation complete.")

In [ ]:
# Find the generated AMReX plotfile directory
plotfile_dirs = [d for d in glob.glob(f"{out_dir}/*") if os.path.isdir(d)]
latest_plotfile = sorted(plotfile_dirs)[-1]

print(f"Loading AMReX data from: {latest_plotfile}")

# Load the data into yt (suppress spammy yt logging)
yt.funcs.mylog.setLevel(40)
ds = yt.load(latest_plotfile)

# The C++ solver names the field 'solution_potential'
field_name = ('boxlib', 'solution_potential')

# Create a Slice Plot through the middle of the Y-axis
slc = yt.SlicePlot(ds, normal='y', fields=field_name)

# Tell yt to use a LINEAR scale, not a log scale (crucial for a 0 to 1 gradient!)
slc.set_log(field_name, False)

# Lock the colorbar limits to our exact boundary conditions
slc.set_zlim(field_name, 0.0, 1.0)

# Make it look good!
slc.set_cmap(field_name, 'magma')
slc.annotate_title("Steady-State Diffusion Potential (Z-Direction)")

# Display inline in the Jupyter Notebook
slc.show()

## 4. Exporting to Battery Models (BPX Format)

OpenImpala provides a native `ResultsJSON` builder in its C++ core that generates files compliant with the **Battery Parameter eXchange (BPX)** standard.

*Note: In continuum physics, Effective Diffusivity is defined as $D_{eff} = D_{bulk} \frac{\varepsilon}{\tau}$. The ratio $\frac{D_{eff}}{D_{bulk}}$ is the **Transport Efficiency**.*

In [ ]:
# Access the low-level C++ bindings via openimpala.core
rj = oic.ResultsJSON()

# Define the physics we just solved (Diffusion)
physics_cfg = oic.PhysicsConfig.from_type_string("diffusion")
rj.set_physics_config(physics_cfg)

# Set metadata
rj.set_input_file(data_path)
rj.set_phase_id(1)
rj.set_volume_fraction(vf.fraction)

# Add our directional results
for d, tau in tau_results.items():
    if tau is not None:
        transport_efficiency = vf.fraction / tau
        rj.add_direction_result(d, transport_efficiency)

# Enable BPX fragment generation for the Positive Electrode
rj.set_bpx_electrode("positive")
rj.set_provenance(sample_id="SampleData_2Phase", provenance_uri="https://github.com/BASE-Laboratory/OpenImpala")

# Build the JSON string and save to disk
bpx_json_string = rj.build_json_string()
bpx_filepath = "openimpala_cathode.json"

with open(bpx_filepath, "w") as f:
    f.write(bpx_json_string)

print(f"Exported BPX data to {bpx_filepath}")

## 5. The Mic Drop: Simulating in PyBaMM

We aren't just calculating numbers in a vacuum. We can now load our newly generated BPX JSON directly into [PyBaMM](https://pybamm.org/) and simulate a battery discharge curve based on the exact geometry of our 3D scan.

This is the bridge between microstructural imaging and device-level performance -- automated by OpenImpala.

In [ ]:
print("Loading PyBaMM DFN model...")
model = pybamm.lithium_ion.DFN()

# Load default generic parameters (Chen 2020 is an industry standard)
param = pybamm.ParameterValues("Chen2020")

# Read our OpenImpala BPX JSON
with open(bpx_filepath, "r") as f:
    oi_data = json.load(f)

# Extract the BPX parameters we just calculated
cathode_params = oi_data["bpx"]["Parameterisation"]["Positive electrode"]
oi_porosity = cathode_params["Porosity"]
oi_bruggeman = cathode_params["Bruggeman coefficient (electrolyte)"]

print("Overriding PyBaMM defaults with OpenImpala 3D scan data:")
print(f"  -> Positive electrode porosity:      {oi_porosity:.4f}")
print(f"  -> Positive electrode Bruggeman exp: {oi_bruggeman:.4f}")

# Inject our real-world parameters into the simulation
param.update({
    "Positive electrode porosity": oi_porosity,
    "Positive electrode Bruggeman coefficient (electrolyte)": oi_bruggeman
}, check_already_exists=False)

print("\nRunning battery discharge simulation...")
sim = pybamm.Simulation(model, parameter_values=param)
solution = sim.solve([0, 3600])  # Simulate a 1-hour discharge

# Extract simulation data for plotting
t = solution["Time [s]"].entries
V = solution["Terminal voltage [V]"].entries

fig, ax = plt.subplots(figsize=(6, 4), dpi=120)
ax.plot(t, V, lw=3, color="#d62728")
ax.set_xlabel("Time (s)", fontweight="bold")
ax.set_ylabel("Terminal Voltage (V)", fontweight="bold")
ax.set_title("PyBaMM Discharge Curve\n(Powered by OpenImpala Microstructure)", fontweight="bold")
ax.grid(True, linestyle="--", alpha=0.6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print("Done.")

## What's Next?

You just turned a raw CT scan into a working battery simulation -- fully automated.

**Continue the journey:**
- [Tutorial 3: REV & Uncertainty](03_rev_and_uncertainty.ipynb) -- How do you know your crop is representative? Prove it statistically.
- [Tutorial 4: Multi-Phase Materials](04_multiphase_and_fields.ipynb) -- Go beyond binary: add Carbon-Binder Domain (CBD) and visualize bottlenecks.
- [Tutorial 7: Laptop to Supercomputer](07_hpc_scaling.ipynb) -- Scale up to analyse massive synchrotron datasets with MPI.